# 04 · High-price episodes and smoothing

Explore real price extremes without deleting them. Smoothing is a descriptive transformation; it never changes the price target or proves that market volatility was reduced. The centered Gaussian filter uses future observations and is explicitly retrospective.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nem_analysis.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from nem_analysis import load_market, filter_market, aggregate_market, smooth_prices, price_episodes, score_forecasts, season_labels
market, metadata = load_market()
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['#087e72', '#df9562', '#8296b0', '#233c35']
print(f"{len(market):,} half-hour positions | {market.index.min()}–{market.index.max()} | AEST (UTC+10)")

44,911 half-hour positions | 2022-01-01 00:00:00–2024-07-24 15:00:00 | AEST (UTC+10)


## Define events using an explicit economic threshold
Episodes join only contiguous half-hours whose mean is at least the threshold. Five-minute extremes can occur inside an otherwise lower-price half-hour.

In [2]:
threshold = 300
events = price_episodes(market, threshold)
display(events.head(15))
print(f'{len(events):,} episodes; {market.price.ge(threshold).sum():,} high-price half-hours; {market.price.dropna().lt(0).mean():.1%} negative-price half-hours.')

,start,end,peak_time,peak_price,peak_dispatch,duration_minutes,demand,solar,temperature
0,2024-05-08 16:00:00,2024-05-08 20:00:00,2024-05-08 17:30:00,16599.910,16600.00,270,9077.0,3.081,17.8
1,2022-06-13 18:00:00,2022-06-13 18:30:00,2022-06-13 18:30:00,15100.000,15100.00,60,11097.0,0.000,NaN
2,2024-05-08 07:00:00,2024-05-08 08:30:00,2024-05-08 07:30:00,14955.918,16432.34,120,8293.0,167.700,14.5
3,2024-05-07 17:00:00,2024-05-07 18:30:00,2024-05-07 18:00:00,14523.075,16599.89,120,9504.0,0.000,15.9
4,2022-05-03 17:00:00,2022-05-03 19:00:00,2022-05-03 18:00:00,13496.323,15100.00,150,9016.0,0.000,19.0
5,2024-02-29 15:00:00,2024-02-29 19:00:00,2024-02-29 17:00:00,13325.677,16600.00,270,13643.0,811.023,28.7
6,2023-03-16 17:00:00,2023-03-16 19:00:00,2023-03-16 18:30:00,9805.300,14506.69,150,11638.0,11.179,31.6
7,2023-05-24 17:30:00,2023-05-24 19:00:00,2023-05-24 18:00:00,9178.597,14141.35,120,10667.0,0.000,15.3
8,2024-01-21 18:00:00,2024-01-21 20:00:00,2024-01-21 19:00:00,8008.105,9569.76,150,12341.0,80.225,29.9
9,2022-05-05 18:00:00,2022-05-05 19:00:00,2022-05-05 18:00:00,7832.568,11382.85,90,9120.0,0.000,18.4


449 episodes; 3,211 high-price half-hours; 4.2% negative-price half-hours.


## Inspect the largest episode with its surrounding days

In [3]:
largest = events.iloc[0]
window = filter_market(market, largest.start.date() - pd.Timedelta(days=1), largest.end.date() + pd.Timedelta(days=1))
window = window.assign(ema=smooth_prices(window,'Trailing EMA',6), gaussian=smooth_prices(window,'Centered Gaussian',6))
fig=go.Figure()
for column,label,color in [('price','Observed','#233c35'),('ema','Trailing EMA','#087e72'),('gaussian','Centered Gaussian · retrospective','#df9562')]:
    fig.add_trace(go.Scatter(x=window.index,y=window[column],name=label,line_color=color,connectgaps=False))
fig.add_hline(y=threshold,line_dash='dash',line_color='#8296b0')
fig.update_layout(template='plotly_white',height=450,yaxis_title='AUD/MWh',xaxis_title='Interval ending · AEST',hovermode='x unified')
fig.show(renderer='plotly_mimetype')

## Prove the EMA does not use future prices
The centered filter is allowed to change earlier values when future prices change, which is precisely why it is excluded from forecasting.

In [4]:
origin = window.index[len(window)//2]
changed = window.copy()
changed.loc[changed.index > origin,'price'] = 999999
original_ema=smooth_prices(window,'Trailing EMA',6)
changed_ema=smooth_prices(changed,'Trailing EMA',6)
pd.testing.assert_series_equal(original_ema.loc[:origin],changed_ema.loc[:origin])
original_gaussian=smooth_prices(window,'Centered Gaussian',6)
changed_gaussian=smooth_prices(changed,'Centered Gaussian',6)
assert not np.allclose(original_gaussian.loc[:origin].fillna(0),changed_gaussian.loc[:origin].fillna(0))
print('EMA passes the future perturbation test. Centered Gaussian correctly demonstrates future dependence.')

EMA passes the future perturbation test. Centered Gaussian correctly demonstrates future dependence.


## Sensitivity to the exploratory threshold
Changing a threshold alters an event definition. It does not retrain the fixed $300/MWh classifier.

In [5]:
sensitivity=[]
for limit in [100,300,1000,5000]:
    sensitivity.append({'threshold_AUD_MWh':limit,'high_half_hours':int(market.price.ge(limit).sum()),'episodes':len(price_episodes(market,limit))})
display(pd.DataFrame(sensitivity))
assert price_episodes(market,5000).peak_price.ge(5000).all()

,threshold_AUD_MWh,high_half_hours,episodes
0,100,20988,2003
1,300,3211,449
2,1000,112,67
3,5000,37,19


## Use the dashboard
Run `.venv/bin/python -m streamlit run streamlit_app.py` to explore these analyses interactively. The React/ECharts interface is also available with `npm run dev`. Both consume the same checked data artifact and preserve unsmoothed targets.